# Mejoras para king-county


Proyecto probable si tienes el CSV descargado antes.

La mejora estrella es evitar fuga temporal: entrenar con ventas antiguas y probar con ventas posteriores.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

DATA_PATH = Path("datasets/king-county/kc_house_data.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../proyects-upgrades/datasets/king-county/kc_house_data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError("Falta kc_house_data.csv. Descárgalo antes de cortar internet.")

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

# No usar id como feature directa: memoriza casas concretas.
df = df.drop(columns=["id"])


In [ ]:
# Feature engineering temporal y de vivienda.
df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month
df["house_age"] = df["sale_year"] - df["yr_built"]
df["was_renovated"] = (df["yr_renovated"] > 0).astype(int)
df["years_since_renovation"] = np.where(df["yr_renovated"] > 0, df["sale_year"] - df["yr_renovated"], 0)
df["living_lot_ratio"] = df["sqft_living"] / df["sqft_lot"]
df["bathrooms_per_bedroom"] = df["bathrooms"] / df["bedrooms"].replace(0, np.nan)

df = df.drop(columns=["date"])


In [ ]:
# Split temporal: pasado para train, futuro para test.
split_idx = int(len(df) * 0.8)
train_set = df.iloc[:split_idx]
test_set = df.iloc[split_idx:]

X_train = train_set.drop(columns=["price"])
y_train = train_set["price"]
X_test = test_set.drop(columns=["price"])
y_test = test_set["price"]

print("Train fechas antiguas -> Test fechas futuras")
print(X_train.shape, X_test.shape)


In [ ]:
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42, n_jobs=-1)),
])

param_grid = {
    "regressor__n_estimators": [100, 200],
    "regressor__max_depth": [12, 20, None],
    "regressor__min_samples_leaf": [1, 3, 5],
}

search = GridSearchCV(
    model,
    param_grid=param_grid,
    cv=TimeSeriesSplit(n_splits=3),
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

search.fit(X_train, y_train)
print("Mejores parámetros:", search.best_params_)


In [ ]:
y_pred = search.best_estimator_.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.0f}")
print(f"MAE:  {mae:.0f}")
print(f"R2:   {r2:.4f}")


Explicación corta:

“Uso split temporal porque en precios de vivienda el futuro no debe contaminar el pasado. Elimino `id` porque memoriza propiedades. Extraigo características temporales y ratios, y uso `TimeSeriesSplit` para ajustar hiperparámetros sin romper el orden temporal.”
